# **Uji Ketahanan Model terhadap Variasi Bahasa Lokal — Pilot Development**

---

**Proyek:** KP Agentic AI + LLM  
**Tahap:** Step 13 — Pilot ketahanan model  
**Dataset:** MASSIVE-Agents (development)  
**Model:** Qwen2.5, Gemma 2, Sahabat-AI, dan LFM2.5

Pilot ini membandingkan prediksi pada instruksi Indonesia formal dan variasi bahasanya. Variasi v4 masih menunggu validasi manusia, sehingga hasil pilot hanya digunakan untuk memeriksa alur eksperimen, **bukan hasil utama penelitian**.

## **1. Konfigurasi Pilot**

---

Pilot menggunakan 4 variasi untuk setiap kategori (20 variasi). Jumlah request adalah 80, karena setiap variasi diuji pada empat model. Data test tidak digunakan. Konfigurasi inferensi sama dengan baseline development.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import sys
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from itertools import combinations
from pathlib import Path
from types import SimpleNamespace

from IPython.display import display
import pandas as pd

SEED_SAMPLE = 20261110
SEED_INFERENCE = 20261020
N_PER_CATEGORY = 4
RUN_NAME = "pilot_20_v1"
API_BASE = "http://127.0.0.1:11434"
TIMEOUT_SECONDS = 1200
KEEP_ALIVE = "5m"
BFCL_VERSION = "2025.12.17"
PROMPT_VERSION = "compact_catalog_v1"
CATEGORIES = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]
MODELS = [
    "qwen2.5:7b-instruct",
    "gemma2:9b-instruct-q4_K_M",
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
    "lfm2.5:8b-a1b-q4_K_M",
]
OPTIONS = {
    "temperature": 0,
    "seed": SEED_INFERENCE,
    "num_ctx": 16384,
    "num_predict": 384,
    "num_batch": 128,
}
print("Python:", sys.version.split()[0])
print("Pilot:", N_PER_CATEGORY, "variasi per kategori")
print("Model:", len(MODELS))

Python: 3.11.9
Pilot: 4 variasi per kategori
Model: 4


## **2. Dataset dan Hasil Baseline**

---

Variasi v4 dibaca dari development set. Skor AST instruksi formal menggunakan hasil baseline empat model yang sudah tersedia, tanpa inferensi ulang. Perbandingan hanya dilakukan pada record induk yang sama.

In [2]:
def locate_project():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"D:\KP-Agentic-AI")]
    for root in candidates:
        if (root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl").is_file():
            return root
    raise FileNotFoundError("Folder project dengan id-ID_development.jsonl tidak ditemukan.")


def read_csv(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_csv(path, rows, columns=None):
    if not rows and not columns:
        raise ValueError("Tidak ada data untuk disimpan")
    columns = columns or list(rows[0])
    temp = path.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(path)


def digest_file(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def as_bool(value):
    return value is True or str(value).strip().lower() == "true"

ROOT = locate_project()
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
V4_DIR = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v4"
BASELINE_DIR = ROOT / "results" / "baseline_evaluation"
AST_OLD_FILE = ROOT / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv"
AST_LFM_FILE = ROOT / "results" / "model_tambahan_lfm25" / "ast_per_prediction_development_full.csv"
CFG_BASE_FILE = BASELINE_DIR / "config_development_full_v1_verified.json"
CFG_LFM_FILE = ROOT / "results" / "model_tambahan_lfm25" / "config_development_full.json"
VARIANTS_FILE = V4_DIR / "sampel_validasi_v4.csv"
OUT = ROOT / "results" / "ketahanan_bahasa_lokal" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

parents_list = read_jsonl(DATA_DIR / "id-ID_development.jsonl")
parents = {r["row_key"]: r for r in parents_list}
variants_all = read_csv(VARIANTS_FILE)
assert len(parents) == 397 and len(variants_all) == 500
assert all(r["split"] == "development" and r["language_code"] == "id-ID" for r in parents_list)
assert all(r["status_validasi"] == "BELUM_VALIDASI_MANUSIA" for r in variants_all)

formal_ast_rows = read_csv(AST_OLD_FILE) + read_csv(AST_LFM_FILE)
formal_ast = {}
for r in formal_ast_rows:
    key = (r["model"], r["row_key"])
    if key in formal_ast:
        raise RuntimeError(f"Duplikat AST baseline: {key}")
    formal_ast[key] = r

assert len(formal_ast) == 979 * 4
cfg_base = json.loads(CFG_BASE_FILE.read_text(encoding="utf-8-sig"))
cfg_lfm = json.loads(CFG_LFM_FILE.read_text(encoding="utf-8-sig"))
print("Induk ID development:", len(parents))
print("Sampel kandidat v4:", len(variants_all))
print("AST formal 4 model:", len(formal_ast))
print("Folder hasil:", OUT)

Induk ID development: 397
Sampel kandidat v4: 500
AST formal 4 model: 3916
Folder hasil: D:\KP-Agentic-AI\results\ketahanan_bahasa_lokal\pilot_20_v1


## **3. Pemilihan Variasi Pilot**

---

Variasi dipilih secara acak dengan seed tetap dan satu `group_id` untuk setiap instruksi dalam pilot. GT, fungsi, isi instruksi induk, dan split dicocokkan kembali dengan dataset sumber. Semua variasi tetap berstatus **belum tervalidasi manusia**.

In [3]:
rng = random.Random(SEED_SAMPLE)
selected = []
used_groups = set()
for category in CATEGORIES:
    candidates = sorted((r for r in variants_all if r["kategori"] == category), key=lambda r: r["variant_id"])
    rng.shuffle(candidates)
    chosen = []
    for row in candidates:
        parent = parents[row["row_key_induk"]]
        if row["group_id"] in used_groups:
            continue
        if row["split"] != parent["split"] or row["split"] != "development":
            raise RuntimeError("Split kandidat berbeda dari sumber")
        if row["group_id"] != parent["group_id"]:
            raise RuntimeError("group_id kandidat berbeda dari sumber")
        if row["instruksi_id_asli"] != parent["question"]:
            raise RuntimeError("Instruksi induk berubah")
        if json.loads(row["gt_json"]) != parent["gt"]:
            raise RuntimeError("GT kandidat berubah")
        if row["fungsi_gt"] != next(iter(parent["gt"])):
            raise RuntimeError("Fungsi GT kandidat berubah")
        if row["instruksi_variasi"].strip() == parent["question"].strip():
            raise RuntimeError("Variasi tidak berbeda dari sumber")
        chosen.append(row)
        used_groups.add(row["group_id"])
        if len(chosen) == N_PER_CATEGORY:
            break
    if len(chosen) != N_PER_CATEGORY:
        raise RuntimeError(f"Sampel kategori {category} kurang dari {N_PER_CATEGORY}")
    selected.extend(chosen)

assert len(selected) == len(set(r["variant_id"] for r in selected)) == N_PER_CATEGORY * len(CATEGORIES)
assert len(set(r["group_id"] for r in selected)) == len(selected)
assert all((m, r["row_key_induk"]) in formal_ast for r in selected for m in MODELS)
print("Total variasi:", len(selected))
print("Jumlah kategori:", dict(Counter(r["kategori"] for r in selected)))
print("Setiap group_id unik: PASS")
display(pd.DataFrame(selected)[["variant_id", "kategori", "instruksi_id_asli", "instruksi_variasi"]].head(10))

Total variasi: 20
Jumlah kategori: {'slang': 4, 'singkatan': 4, 'typo': 4, 'code_switch_id_en': 4, 'code_switch_id_jawa': 4}
Setiap group_id unik: PASS


,variant_id,kategori,instruksi_id_asli,instruksi_variasi
0,id-ID:0000195::slang,slang,bangunkan saya dalam setengah jam,bangunkan aku dalam setengah jam
1,id-ID:0000704::slang,slang,apakah saya memiliki rapat pada hari sabtu dua...,apakah aku memiliki rapat pada hari sabtu dua ...
2,id-ID:0000452::slang,slang,bagaimana cuaca pekan ini,gimana cuaca pekan ini
3,id-ID:0000694::slang,slang,informasi apa yang harus saya ketahui untuk ra...,informasi apa yang harus aku ketahui untuk rap...
4,id-ID:0000932::singkatan,singkatan,beri tahu saya acara kontes pertunjukan anjing...,beri tahu saya acara kontes pertunjukan anjing...
5,id-ID:0000085::singkatan,singkatan,tolong hapus alarm saya,tlg hapus alarm saya
6,id-ID:0001243::singkatan,singkatan,balas cepat ke jack jadi tolong kirim sebuah e...,balas cepat ke jack jadi tlg kirim sebuah email
7,id-ID:0000357::singkatan,singkatan,apakah alarm pengingat saya sudah disetel untu...,apkh alarm pengingat saya sudah disetel untuk ...
8,id-ID:0000181::typo,typo,atur alarm untuk pukul enam tiga puluh pagi,atur alarm utnuk pukul enam tiga puluh pagi
9,id-ID:0000066::typo,typo,matikan alarm jam delapan pagi,mtaikan alarm jam delapan pagi


## **4. Prompt Baseline dan Konfigurasi Model**

---

Prompt sama dengan `compact_catalog_v1`. Hanya teks permintaan pengguna yang diganti dengan variasi. Format API Sahabat-AI tetap mengikuti pengujian baseline lama, sedangkan tiga model lainnya menggunakan `chat_json`.

In [4]:
def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def compact_catalog(record):
    lines = []
    for func in record["function"]:
        description = func.get("description", "")
        description = description.split("The default user experience")[0]
        description = description.split(". ")[0].strip()[:100]
        properties = func.get("parameters", {}).get("properties", {})
        arguments = ", ".join(properties.keys())
        lines.append(f"{func['name']}: {description}\n  Arguments: {arguments if arguments else 'none'}")
    return "\n".join(lines)


def compact_prompt(record):
    return (
        "Select the most appropriate function for the user request.\n"
        "Return one JSON object with exactly two keys: name and arguments.\n"
        "The name must be a function from the catalog.\n"
        "Arguments must contain the relevant parameter names and string values.\n"
        "Do not copy function definitions or include explanations.\n\n"
        "Available functions:\n"
        + compact_catalog(record)
        + "\n\nUser request:\n"
        + record["question"]
        + '\n\nJSON output: {"name":"function_name","arguments":{}}'
    )


def calc_hashes(records):
    data_digest = hashlib.sha256()
    prompt_digest = hashlib.sha256()
    for record in records:
        data_digest.update(compact_json({
            "row_key": record["row_key"], "question": record["question"],
            "function": record["function"], "gt": record["gt"],
        }).encode("utf-8"))
        data_digest.update(b"\n")
        prompt_digest.update(compact_prompt(record).encode("utf-8"))
        prompt_digest.update(b"\n")
    return data_digest.hexdigest(), prompt_digest.hexdigest()

# Kecocokan baseline diperiksa pada 979 development penuh.
all_development = read_jsonl(DATA_DIR / "en-US_development.jsonl") + parents_list
all_ordered = sorted(all_development, key=lambda r: (str(r["source_record_id"]), r["language_code"]))
expected_data_sha, expected_prompt_sha = calc_hashes(all_ordered)
assert expected_data_sha == cfg_base["dataset_sha256"]
assert expected_prompt_sha == cfg_base["prompt_sha256"]
assert cfg_lfm["prompt_sha256"] == expected_prompt_sha
assert cfg_lfm["dataset_sha256"] == expected_data_sha

adapted = {}
for row in selected:
    record = dict(parents[row["row_key_induk"]])
    record["question"] = row["instruksi_variasi"]
    adapted[row["variant_id"]] = record
assert all(len(record["function"]) == 55 for record in adapted.values())
print("Hash prompt baseline verified: PASS")
print("Katalog fungsi per record: 55")

Hash prompt baseline verified: PASS
Katalog fungsi per record: 55


## **5. Pemeriksaan Ollama dan BFCL**

---

Identitas dan digest model dicocokkan dengan eksperimen baseline. BFCL AST menggunakan versi evaluator yang sama, yaitu `2025.12.17`. Perbedaan identitas model menghentikan proses agar hasilnya tidak dicampur.

In [5]:
def ollama_request(endpoint, payload=None, timeout=TIMEOUT_SECONDS):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        API_BASE + endpoint, data=data,
        headers={"Content-Type": "application/json"},
        method="POST" if data is not None else "GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc

try:
    tags = ollama_request("/api/tags", timeout=20)
except Exception as exc:
    raise RuntimeError("Ollama lokal tidak dapat diakses") from exc

installed = {m.get("name"): m for m in tags.get("models", [])}
missing = [m for m in MODELS if m not in installed]
if missing:
    raise RuntimeError(f"Model belum terpasang: {missing}")

expected_digests = dict(cfg_base["model_digests"])
expected_digests[MODELS[-1]] = cfg_lfm["model_digest"]
for model in MODELS:
    if installed[model].get("digest") != expected_digests[model]:
        raise RuntimeError(f"Digest model berubah dari baseline: {model}")

try:
    current_bfcl = version("bfcl-eval")
except PackageNotFoundError as exc:
    raise RuntimeError("bfcl-eval belum terpasang pada kernel ini") from exc
if current_bfcl != BFCL_VERSION:
    raise RuntimeError(f"Versi BFCL berbeda: {current_bfcl}")

from bfcl_eval.eval_checker.ast_eval.ast_checker import ast_checker
from bfcl_eval.constants.enums import Language
from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING

for model in MODELS:
    MODEL_CONFIG_MAPPING[model] = SimpleNamespace(underscore_to_dot=False)

print("Model cocok dengan baseline: PASS")
print("BFCL AST:", current_bfcl)

Model cocok dengan baseline: PASS
BFCL AST: 2025.12.17


## **6. Inferensi Variasi Bahasa**

---

Keempat model memproses 20 variasi dengan prompt dan opsi yang sama seperti baseline. Prediksi disimpan setelah setiap request agar proses dapat dilanjutkan tanpa mengulang hasil yang sudah tersimpan. Hasil lama tetap terpisah.

In [6]:
def build_request(model, record):
    content = compact_prompt(record)
    payload = {"model": model, "stream": False, "options": dict(OPTIONS), "keep_alive": KEEP_ALIVE}
    if model.startswith("Supa-AI/"):
        payload["prompt"] = (
            "<start_of_turn>user\n" + content
            + "\n<end_of_turn>\n<start_of_turn>model\n"
        )
        payload["raw"] = True
        payload["options"]["stop"] = ["<end_of_turn>", "<eos>"]
        return "/api/generate", payload
    payload["messages"] = [{"role": "user", "content": content}]
    payload["format"] = "json"
    return "/api/chat", payload


def parse_model_json(raw):
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None, "invalid_json"
    if not isinstance(obj, dict):
        return None, "invalid_structure"
    if not isinstance(obj.get("name"), str) or not isinstance(obj.get("arguments"), dict):
        return None, "invalid_structure"
    return obj, "parsed"

identity = {
    "run": RUN_NAME,
    "source_v4_sha256": digest_file(VARIANTS_FILE),
    "source_id_sha256": digest_file(DATA_DIR / "id-ID_development.jsonl"),
    "selected_variant_ids": [r["variant_id"] for r in selected],
    "models": MODELS,
    "model_digests": {m: installed[m].get("digest", "") for m in MODELS},
    "prompt_baseline_sha256": expected_prompt_sha,
    "options": OPTIONS,
    "seed_sample": SEED_SAMPLE,
    "bfcl_eval": BFCL_VERSION,
    "validation_state": "unvalidated_exploratory_pilot",
}
fingerprint = hashlib.sha256(compact_json(identity).encode("utf-8")).hexdigest()[:20]
CONFIG_FILE = OUT / "config_pilot.json"
SELECTED_FILE = OUT / "selected_variants_pilot.csv"
PRED_FILE = OUT / "predictions_variants_pilot.csv"
AST_FILE = OUT / "ast_variants_pilot.csv"
PAIRED_FILE = OUT / "paired_comparison_pilot.csv"
SUMMARY_FILE = OUT / "summary_pilot.csv"
META_FILE = OUT / "metadata_pilot.json"

if CONFIG_FILE.exists():
    previous = json.loads(CONFIG_FILE.read_text(encoding="utf-8-sig"))
    if previous.get("fingerprint") != fingerprint:
        raise RuntimeError("Konfigurasi pilot berbeda dari hasil tersimpan. Data lama tidak diubah")
if not CONFIG_FILE.exists():
    if PRED_FILE.exists():
        raise RuntimeError("File prediksi sudah ada tanpa config. Data tidak diubah")
    CONFIG_FILE.write_text(json.dumps({**identity, "fingerprint": fingerprint}, indent=2, ensure_ascii=False), encoding="utf-8")

write_csv(SELECTED_FILE, selected)
FIELDS = [
    "fingerprint", "model", "variant_id", "row_key_induk", "group_id", "kategori",
    "instruksi_id_asli", "instruksi_variasi", "predicted_function",
    "prediction_arguments_json", "parsed_json", "done_reason", "prompt_tokens",
    "output_tokens", "latency_seconds", "raw_output", "timestamp_utc",
]
existing = read_csv(PRED_FILE) if PRED_FILE.exists() else []
if existing and list(existing[0]) != FIELDS:
    raise RuntimeError("Kolom prediksi tersimpan tidak sesuai")
completed = set()
for row in existing:
    key = (row["model"], row["variant_id"])
    if key in completed or row["fingerprint"] != fingerprint:
        raise RuntimeError("Duplikasi atau fingerprint prediksi tidak sesuai")
    completed.add(key)
print("Fingerprint:", fingerprint)
print("Tersimpan:", len(completed), "dari", len(MODELS) * len(selected))

for model in MODELS:
    print("MODEL:", model)
    for i, row in enumerate(selected, 1):
        key = (model, row["variant_id"])
        if key in completed:
            continue
        record = adapted[row["variant_id"]]
        endpoint, payload = build_request(model, record)

        # Gunakan CPU untuk model berbasis Gemma 2 9B
        if model in [
            "gemma2:9b-instruct-q4_K_M",
            "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m"
        ]:
            payload["options"]["num_gpu"] = 0

        # Kosongkan memori model setelah inferensi
        payload["keep_alive"] = 0

        started = time.perf_counter()
        response = ollama_request(endpoint, payload)
        raw = response.get("response", "") if endpoint == "/api/generate" else (response.get("message") or {}).get("content", "")
        prediction, parsed = parse_model_json(raw)
        item = {
            "fingerprint": fingerprint,
            "model": model,
            "variant_id": row["variant_id"],
            "row_key_induk": row["row_key_induk"],
            "group_id": row["group_id"],
            "kategori": row["kategori"],
            "instruksi_id_asli": row["instruksi_id_asli"],
            "instruksi_variasi": row["instruksi_variasi"],
            "predicted_function": prediction["name"] if parsed == "parsed" else "",
            "prediction_arguments_json": compact_json(prediction["arguments"]) if parsed == "parsed" else "",
            "parsed_json": parsed == "parsed",
            "done_reason": response.get("done_reason", ""),
            "prompt_tokens": response.get("prompt_eval_count", ""),
            "output_tokens": response.get("eval_count", ""),
            "latency_seconds": round(time.perf_counter() - started, 2),
            "raw_output": raw,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
        with PRED_FILE.open("a", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=FIELDS)
            if handle.tell() == 0:
                writer.writeheader()
            writer.writerow(item)
        existing.append(item)
        completed.add(key)
        if i == 1 or i % 5 == 0 or i == len(selected):
            print(f"  {i}/{len(selected)} | {row['kategori']} | {parsed}")

if len(existing) != len(MODELS) * len(selected):
    raise RuntimeError("Prediksi belum lengkap")
print("Inferensi selesai:", len(existing), "prediksi variasi")

Fingerprint: e5660a3f3d9d1a595dc6
Tersimpan: 20 dari 80
MODEL: qwen2.5:7b-instruct
MODEL: gemma2:9b-instruct-q4_K_M
  1/20 | slang | parsed
  5/20 | singkatan | parsed
  10/20 | typo | parsed
  15/20 | code_switch_id_en | parsed
  20/20 | code_switch_id_jawa | parsed
MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
  1/20 | slang | parsed
  5/20 | singkatan | parsed
  10/20 | typo | parsed
  15/20 | code_switch_id_en | parsed
  20/20 | code_switch_id_jawa | parsed
MODEL: lfm2.5:8b-a1b-q4_K_M
  1/20 | slang | parsed
  5/20 | singkatan | parsed
  10/20 | typo | parsed
  15/20 | code_switch_id_en | parsed
  20/20 | code_switch_id_jawa | parsed
Inferensi selesai: 80 prediksi variasi


## **7. Evaluasi BFCL AST**

---

Prediksi variasi dinilai dengan checker BFCL AST resmi, menggunakan GT dan katalog fungsi dari instruksi induk. Skor ini hanya untuk pemeriksaan pilot dan belum menjadi skor evaluasi penelitian karena dataset variasi masih menunggu validasi manusia.

In [8]:
from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING

bfcl_checker_name = next(
    name for name, config in MODEL_CONFIG_MAPPING.items()
    if "_" not in name and not config.underscore_to_dot
)

print("Konfigurasi AST checker:", bfcl_checker_name)

ast_results = []
for pred in existing:
    parent = parents[pred["row_key_induk"]]
    ok = False
    error_type = ""
    error_detail = ""
    if as_bool(pred["parsed_json"]):
        args = json.loads(pred["prediction_arguments_json"])
        result = ast_checker(
            func_description=parent["function"],
            model_output=[{pred["predicted_function"]: args}],
            possible_answer=[parent["gt"]],
            language=Language.PYTHON,
            test_category="multiple_function",
            model_name=bfcl_checker_name,
        )
        if not isinstance(result, dict) or not isinstance(result.get("valid"), bool):
            raise RuntimeError(f"Respons AST tidak valid: {result!r}")
        ok = result["valid"]
        error_type = "" if ok else str(result.get("error_type", "checker_error"))
        error_detail = json.dumps(result.get("error", []), ensure_ascii=False)
    else:
        error_type = "adapter:invalid_json"
        error_detail = "Prediksi JSON tidak terbaca"
    formal = formal_ast[(pred["model"], pred["row_key_induk"])]
    ast_results.append({
        "model": pred["model"],
        "variant_id": pred["variant_id"],
        "row_key_induk": pred["row_key_induk"],
        "group_id": pred["group_id"],
        "kategori": pred["kategori"],
        "ast_formal_correct": as_bool(formal["bfcl_ast_correct"]),
        "ast_variasi_correct": ok,
        "fungsi_formal": formal["prediction_function"],
        "fungsi_variasi": pred["predicted_function"],
        "fungsi_gt": next(iter(parent["gt"])),
        "ast_error_type": error_type,
        "ast_error_message": error_detail,
        "status_validasi": "BELUM_VALIDASI_MANUSIA",
    })

assert len(ast_results) == 80
write_csv(AST_FILE, ast_results)
print("AST formal benar:", sum(x["ast_formal_correct"] for x in ast_results))
print("AST variasi benar:", sum(x["ast_variasi_correct"] for x in ast_results))

Konfigurasi AST checker: gorilla-openfunctions-v2
AST formal benar: 7
AST variasi benar: 6


## **8. Perbandingan Formal dan Variasi**

---

Perbandingan dilakukan pada instruksi induk yang sama untuk setiap model. Perubahan akurasi dicatat per kategori. Pilot berukuran kecil dan variasinya belum tervalidasi, sehingga tidak digunakan untuk menyimpulkan signifikansi statistik.

In [9]:
paired_results = []
for row in ast_results:
    before = row["ast_formal_correct"]
    after = row["ast_variasi_correct"]
    paired_results.append({
        **row,
        "perubahan": ("tetap_benar" if before and after else
                      "turun" if before and not after else
                      "naik" if not before and after else "tetap_salah"),
    })
write_csv(PAIRED_FILE, paired_results)

summary = []
for (model, category) in [(m, c) for m in MODELS for c in CATEGORIES]:
    subset = [r for r in paired_results if r["model"] == model and r["kategori"] == category]
    n = len(subset)
    correct_formal = sum(r["ast_formal_correct"] for r in subset)
    correct_variant = sum(r["ast_variasi_correct"] for r in subset)
    drops = sum(r["perubahan"] == "turun" for r in subset)
    gains = sum(r["perubahan"] == "naik" for r in subset)
    summary.append({
        "model": model,
        "kategori": category,
        "n": n,
        "ast_formal_correct": correct_formal,
        "ast_variasi_correct": correct_variant,
        "ast_formal_pct": round(100 * correct_formal / n, 2),
        "ast_variasi_pct": round(100 * correct_variant / n, 2),
        "delta_pp": round(100 * (correct_variant - correct_formal) / n, 2),
        "turun": drops,
        "naik": gains,
        "status": "EKSPLORATIF_BELUM_VALIDASI_MANUSIA",
    })
write_csv(SUMMARY_FILE, summary)
print("Ringkasan pilot per model/kategori:")
display(pd.DataFrame(summary)[["model", "kategori", "n", "ast_formal_pct", "ast_variasi_pct", "delta_pp"]])

Ringkasan pilot per model/kategori:


,model,kategori,n,ast_formal_pct,ast_variasi_pct,delta_pp
0,qwen2.5:7b-instruct,slang,4,0.0,0.0,0.0
1,qwen2.5:7b-instruct,singkatan,4,0.0,0.0,0.0
2,qwen2.5:7b-instruct,typo,4,0.0,0.0,0.0
3,qwen2.5:7b-instruct,code_switch_id_en,4,75.0,50.0,-25.0
4,qwen2.5:7b-instruct,code_switch_id_jawa,4,0.0,0.0,0.0
5,gemma2:9b-instruct-q4_K_M,slang,4,0.0,0.0,0.0
6,gemma2:9b-instruct-q4_K_M,singkatan,4,25.0,25.0,0.0
7,gemma2:9b-instruct-q4_K_M,typo,4,0.0,0.0,0.0
8,gemma2:9b-instruct-q4_K_M,code_switch_id_en,4,25.0,0.0,-25.0
9,gemma2:9b-instruct-q4_K_M,code_switch_id_jawa,4,0.0,0.0,0.0


## **9. Metadata dan Batasan**

---

Metadata mencatat identitas eksperimen, hash data, model, dan versi evaluator. Hasil pilot belum dapat digunakan sebagai bukti ketahanan model terhadap ragam bahasa lokal sampai validasi semantik dan kewajaran bahasa diselesaikan. Analisis utama nantinya menggunakan pasangan terkontrol yang telah disetujui manusia.

In [10]:
metadata = {
    "run": "step13_pilot_variation_v1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "split": "development",
    "test_set_used": False,
    "models": MODELS,
    "selected_variants": len(selected),
    "predictions": len(existing),
    "categories": dict(Counter(r["kategori"] for r in selected)),
    "unique_group_ids": len(used_groups),
    "source_variant_sha256": digest_file(VARIANTS_FILE),
    "source_id_sha256": digest_file(DATA_DIR / "id-ID_development.jsonl"),
    "baseline_prompt_sha256": expected_prompt_sha,
    "bfcl_version": BFCL_VERSION,
    "status": "exploratory_unvalidated",
    "human_validation": "pending",
    "ground_truth_modified": False,
    "notes": [
        "Perubahan skor hanya deskriptif untuk pilot.",
        "Tidak ada uji signifikansi pada sampel yang belum divalidasi.",
        "Instruksi formal memakai AST baseline sebelumnya, bukan inferensi ulang.",
        "Sahabat-AI memakai raw Gemma 2 prompt sama seperti baseline.",
    ],
}
META_FILE.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
for name in [CONFIG_FILE, SELECTED_FILE, PRED_FILE, AST_FILE, PAIRED_FILE, SUMMARY_FILE, META_FILE]:
    print(name.name)
print("Status: pilot eksploratif, menunggu validasi manusia")

config_pilot.json
selected_variants_pilot.csv
predictions_variants_pilot.csv
ast_variants_pilot.csv
paired_comparison_pilot.csv
summary_pilot.csv
metadata_pilot.json
Status: pilot eksploratif, menunggu validasi manusia


In [11]:
metadata["runtime_overrides"] = {
    "gemma2:9b-instruct-q4_K_M": {
        "num_gpu": 0,
        "keep_alive": 0
    },
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m": {
        "num_gpu": 0,
        "keep_alive": 0
    },
    "lfm2.5:8b-a1b-q4_K_M": {
        "num_gpu": "Ollama otomatis",
        "keep_alive": 0
    },
    "qwen2.5:7b-instruct": {
        "num_gpu": "Ollama otomatis",
        "keep_alive": "5m pada prediksi yang tersimpan sebelum revisi"
    }
}

metadata["bfcl_checker_config"] = bfcl_checker_name

META_FILE.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Metadata berhasil diperbarui.")

Metadata berhasil diperbarui.


In [13]:
import pandas as pd
from pathlib import Path

root = Path(r"D:\KP-Agentic-AI")

file_asli = next(
    p for p in root.rglob("predictions_development_full.csv")
    if "model_tambahan_lfm25" in str(p)
)

file_variasi = next(root.rglob("predictions_variants_pilot.csv"))

asli = pd.read_csv(file_asli)
variasi = pd.read_csv(file_variasi)

model = "lfm2.5:8b-a1b-q4_K_M"

a = asli[
    (asli["model"] == model) &
    (asli["row_key"] == "id-ID:0000085")
].iloc[0]

b = variasi[
    (variasi["model"] == model) &
    (variasi["variant_id"] == "id-ID:0000085::singkatan")
].iloc[0]

print("INSTRUKSI ASLI")
print("tolong hapus alarm saya")
print(a["raw_output"])

print("\nINSTRUKSI SINGKATAN")
print("tlg hapus alarm saya")
print(b["raw_output"])

INSTRUKSI ASLI
tolong hapus alarm saya
{
  "name": "alarm.remove",
  "arguments": {
    "date": "",
    "time": ""
  }
}

INSTRUKSI SINGKATAN
tlg hapus alarm saya
{
  "name": "alarm.query",
  "arguments": {
    "alarm_type": "saya"
  }
}
